In [1]:
def pl_true(sentence, model):
    if isinstance(sentence, str):
        return model[sentence]

    operator = sentence[0]

    if operator == 'NOT':
        return not pl_true(sentence[1], model)
    elif operator == 'AND':
        return all(pl_true(s, model) for s in sentence[1:])
    elif operator == 'OR':
        return any(pl_true(s, model) for s in sentence[1:])
    elif operator == 'IMPLIES':
        return (not pl_true(sentence[1], model)) or pl_true(sentence[2], model)
    raise ValueError(f"Unknown operator: {operator}")

def tt_entails(KB, alpha):
    def get_symbols(expr):
        if isinstance(expr, str):
            return {expr}
        symbols = set()
        for arg in expr[1:]:
            symbols.update(get_symbols(arg))
        return symbols

    symbols = list(get_symbols(KB) | get_symbols(alpha))
    return tt_check_all(KB, alpha, symbols, {})

def tt_check_all(KB, alpha, symbols, model):
    if not symbols:
        if pl_true(KB, model):
            return pl_true(alpha, model)
        else:
            return True
    else:
        P = symbols[0]
        rest = symbols[1:]

        model_true = model.copy()
        model_true[P] = True

        model_false = model.copy()
        model_false[P] = False

        return (tt_check_all(KB, alpha, rest, model_true) and
                tt_check_all(KB, alpha, rest, model_false))

KB = ('AND',
      ('IMPLIES', 'Q', 'P'),
      ('IMPLIES', 'P', ('NOT', 'Q')),
      ('OR', 'Q', 'R'))

print(f"{'P':<6}{'Q':<6}{'R':<6}{'Q->P':<8}{'P->~Q':<8}{'QvR':<8}{'KB':<8}")
print("-" * 50)

for p_val in [True, False]:
    for q_val in [True, False]:
        for r_val in [True, False]:
            m = {'P': p_val, 'Q': q_val, 'R': r_val}
            s1 = pl_true(('IMPLIES', 'Q', 'P'), m)
            s2 = pl_true(('IMPLIES', 'P', ('NOT', 'Q')), m)
            s3 = pl_true(('OR', 'Q', 'R'), m)
            kb_val = pl_true(KB, m)
            marker = " <- KB is TRUE" if kb_val else ""
            print(f"{str(p_val):<6}{str(q_val):<6}{str(r_val):<6}"
                  f"{str(s1):<8}{str(s2):<8}{str(s3):<8}{str(kb_val):<8}{marker}")

print("\n" + "="*50 + "\n")
print(f"ii) Does KB entail R? -> {tt_entails(KB, 'R')}")
print(f"iii) Does KB entail R -> P? -> {tt_entails(KB, ('IMPLIES', 'R', 'P'))}")
print(f"iv) Does KB entail Q -> R? -> {tt_entails(KB, ('IMPLIES', 'Q', 'R'))}")


P     Q     R     Q->P    P->~Q   QvR     KB      
--------------------------------------------------
True  True  True  True    False   True    False   
True  True  False True    False   True    False   
True  False True  True    True    True    True     <- KB is TRUE
True  False False True    True    False   False   
False True  True  False   True    True    False   
False True  False False   True    True    False   
False False True  True    True    True    True     <- KB is TRUE
False False False True    True    False   False   


ii) Does KB entail R? -> True
iii) Does KB entail R -> P? -> False
iv) Does KB entail Q -> R? -> True
